In [1]:
!pip install langchain_google_genai
!pip install -U langchain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.5/471.5 kB 32.7 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.79
    Uninstalling langchain-core-0.3.79:
      Successfully uninstalled langchain-core-0.3.79
  Attempting uninstall: google-ai-generativelanguage
    Found existing installation: google-ai-generativelanguage 0.6.15
    Uninstalling google-ai-generativelanguage-0.6.15:
      Successfully uninstalled google-ai-generativelanguage-0.6.15
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-generativeai 0.8.5 requires google-ai-generativelanguage==0.6.15, but you have google-ai-generativelanguage 0.9.0 which is incompatible.
langchain 0.3.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.7/93.7 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.3/208.3 kB 18.7 MB/s eta 0:00:00
  Attempting uninstall: langchain
    Found existing installation: langchain 0.3.27
    Uninstalling langchain-0.3.27:
      Successfully uninstalled langchain-0.3.27


In [2]:
api_key = "Add your api-key"
# GET YOURS AT https://aistudio.google.com/api-keys
# openrouter
# avalai

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI

MODEL_NAME = "gemini-2.5-flash-preview-05-20"

gemini_chat = ChatGoogleGenerativeAI(model=MODEL_NAME, temperature=0, api_key=api_key)


In [4]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

In [5]:
from langchain.agents import create_agent

In [6]:
agent = create_agent(model=gemini_chat, tools=[])

## Tools

In [8]:
from langchain.tools import tool
from langchain.agents import create_agent
from collections import deque
import time

@tool
def solve_fence_problem(input_data: str) -> str:
    """
    این ابزار مسئله 'Cutting a Fence' (رنگ کردن نرده‌ها) را حل می‌کند.
    ورودی باید دقیقاً فرمت استاندارد مسئله باشد:
    - خط اول: تعداد نرده‌ها (n)
    - خط دوم: ارتفاع نرده‌ها با فاصله (a1 a2 ...)
    - خط سوم: تعداد سوالات (m)
    - خط چهارم: عرض‌های مختلف برای محاسبه میانگین (k1 k2 ...)

    خروجی شامل نام الگوریتم، زمان اجرا و جواب‌های محاسبه شده است.
    """
    start_time = time.time()

    try:
        # 1. پردازش ورودی‌ها (Parsing)
        lines = input_data.strip().split('\n')

        lines = [l for l in lines if l.strip()]

        if len(lines) < 4:
            return "Error: Input format is incorrect. Need 4 lines of data."

        n = int(lines[0].strip())
        heights = list(map(int, lines[1].strip().split()))
        m = int(lines[2].strip())
        k_values = list(map(int, lines[3].strip().split()))

        results = []

        # 2. منطق الگوریتم (Sliding Window Minimum using Deque)
        for k in k_values:
            if k > n:
                results.append(0.0)
                continue

            window_min_sum = 0
            dq = deque()


            for i in range(n):

                if dq and dq[0] < i - k + 1:
                    dq.popleft()


                while dq and heights[dq[-1]] >= heights[i]:
                    dq.pop()

                dq.append(i)


                if i >= k - 1:
                    window_min_sum += heights[dq[0]]



            num_windows = n - k + 1
            expected_value = window_min_sum / num_windows
            results.append(f"{expected_value:.10f}")

        end_time = time.time()
        duration = end_time - start_time

        output_str = " ".join(results)
        return (
            f"Algorithm Used: Sliding Window Minimum (Deque Method)\n"
            f"Execution Time: {duration:.6f} seconds\n"
            f"Final Output:\n{output_str}"
        )

    except Exception as e:
        return f"Error processing data: {str(e)}"

print("Tool 'solve_fence_problem' created successfully!")

Tool 'solve_fence_problem' created successfully!


In [9]:
from langchain.tools import tool
from functools import lru_cache
import time

@tool
def solve_double_elimination(input_data: str) -> str:
    """
    این ابزار، مسئله‌ی Codeforces 1310B - Double Elimination را حل می‌کند.

    فرمت ورودی:
    - خط اول: n k
      (تعداد تیم‌ها = 2^n ، تعداد تیم‌های مورد علاقه = k)
    - خط دوم (اگر k > 0): k عددِ متمایز a_i (شماره‌ی تیم‌های محبوب، بین 1 و 2^n)

    خروجی:
    - یک عدد صحیح: حداکثر تعداد بازی‌هایی که حداقل یک تیم محبوب در آن حضور دارد.

    این پیاده‌سازی همان DP/DFS معروف (حالت بازه‌ای + ماسک ۲ بیتی) است.
    """
    start_time = time.time()

    # --- 1. خواندن و پارس ورودی ---
    lines = [l.strip() for l in input_data.strip().splitlines() if l.strip()]
    if not lines:
        return "Error: empty input."

    try:
        n, k = map(int, lines[0].split())
    except ValueError:
        return "Error: first line must contain two integers n and k."

    total_teams = 1 << n  # 2^n

    fav_indices = []
    if k > 0:
        if len(lines) < 2:
            return "Error: expected a second line with favourite team indices."
        fav_indices = list(map(int, lines[1].split()))
        if len(fav_indices) != k:
            return "Error: number of favourite team indices does not match k."


    fav = [False] * (total_teams + 2)
    for x in fav_indices:
        if 1 <= x <= total_teams:
            fav[x] = True
        else:
            return f"Error: favourite index {x} is out of range 1..{total_teams}."

    INF_NEG = -10**9

    @lru_cache(maxsize=None)
    def dfs(l: int, r: int, s: int) -> int:
        """
        dfs(l, r, s):
        - بازه‌ی [l, r] شامل 2^t تیم است.
        - s ماسک ۲ بیتی است:
          s & 1 : برنده‌ی upper در این بازه محبوب است؟
          s & 2 : برنده‌ی lower در این بازه محبوب است؟ (وزن ۲)
        - خروجی: حداکثر تعداد بازی‌هایی که در *داخل همین زیرتورمنت* و زیرشاخه‌هایش
          با حضور تیم‌های محبوب رخ می‌دهد، اگر وضعیت نهایی این بازه با ماسک s مطابقت داشته باشد.
        """
        if r == l + 1:
            num_fav_teams = int(fav[l]) + int(fav[r])
            num_fav_state = (s & 1) + (s >> 1)

            if num_fav_teams == num_fav_state:

                return min(1, num_fav_state)
            else:

                return INF_NEG

        mid = (l + r) // 2
        best = INF_NEG


        for i in range(4):
            for j in range(4):

                ilose = ((i & 2) | (i << 1) | j) & 3
                jlose = ((j & 2) | (j << 1) | i) & 3


                if ilose == s or jlose == s:
                    val_left = dfs(l, mid, i)
                    val_right = dfs(mid + 1, r, j)
                    if val_left == INF_NEG or val_right == INF_NEG:
                        continue
                    candidate = val_left + val_right
                    if candidate > best:
                        best = candidate

        if best == INF_NEG:
            return INF_NEG


        return best + s


    ans = 0
    for s in range(4):
        val = dfs(1, total_teams, s)
        if val > ans:
            ans = val


    if k > 0:
        ans += 1

    duration = time.time() - start_time
    return (
        "Algorithm Used: Interval DP + Memoized DFS for Double Elimination\n"
        f"Execution Time: {duration:.6f} seconds\n"
        f"Max Games with Favourite Teams: {ans}"
    )


In [10]:
from langchain.tools import tool
import time
import sys

sys.setrecursionlimit(200000)

@tool
def solve_virus_problem(input_data: str) -> str:
    """
    نسخه اصلاح شده (Fix Bug): حل مسئله Virus با مدیریت صحیح kهای بزرگ.
    """
    start_time = time.time()

    # --- 1. پارس کردن ورودی ---
    raw_lines = [l.strip() for l in input_data.strip().splitlines() if l.strip()]
    if not raw_lines: return "Error: Empty input"

    start_index = 0
    n, q, k = 0, 0, 0
    header_found = False
    for i, line in enumerate(raw_lines):
        parts = line.split()
        if len(parts) == 3 and all(p.isdigit() for p in parts):
            try:
                n, q, k = map(int, parts)
                start_index = i + 1
                header_found = True
                break
            except: continue

    if not header_found: return "Error: Header not found"

    data_lines = raw_lines[start_index:]
    queries = [None] * (q + 1)
    op_day = [0] * (q + 1)
    day_end = [0] * (q + k + 10)

    current_day = 1
    line_idx = 0

    try:
        for op in range(1, q + 1):
            if line_idx >= len(data_lines): break
            parts = data_lines[line_idx].split()
            line_idx += 1

            t = int(parts[0])
            op_day[op] = current_day
            day_end[current_day] = op

            if t == 1:
                queries[op] = (1, int(parts[1]), int(parts[2]))
            elif t == 2:
                queries[op] = (2, int(parts[1]), 0)
            elif t == 3:
                queries[op] = (3, 0, 0)
                current_day += 1
    except:
        return "Error parsing queries"

    last_active_day = current_day
    if day_end[current_day] == 0:
        last_active_day = current_day - 1

    seg = [[] for _ in range(4 * (q + 1))]

    def add_edge(node, l, r, ql, qr, u, v):
        if ql > r or qr < l: return
        if ql <= l and r <= qr:
            seg[node].append((u, v))
            return
        mid = (l + r) // 2
        add_edge(node * 2, l, mid, ql, qr, u, v)
        add_edge(node * 2 + 1, mid + 1, r, ql, qr, u, v)

    for op in range(1, q + 1):
        if not queries[op]: continue
        t, x, y = queries[op]
        if t == 1:
            d = op_day[op]
            target_end_day = d + k - 1


            if target_end_day >= last_active_day:
                R = q
            else:
                R = day_end[target_end_day]

            if R == 0: R = q

            L = op
            if L <= R:
                add_edge(1, 1, q, L, R, x, y)

    parent = list(range(n + 1))
    size = [1] * (n + 1)
    history = []

    def find(i):
        while i != parent[i]: i = parent[i]
        return i

    def union(i, j):
        root_i, root_j = find(i), find(j)
        if root_i != root_j:
            if size[root_i] < size[root_j]: root_i, root_j = root_j, root_i
            parent[root_j] = root_i
            size[root_i] += size[root_j]
            history.append((root_j, root_i))
            return True
        history.append(None)
        return False

    def rollback(steps):
        for _ in range(steps):
            op = history.pop()
            if op:
                child, root = op
                parent[child] = child
                size[root] -= size[child]

    ans = []
    def dfs(node, l, r):
        ops = 0
        for u, v in seg[node]:
            if union(u, v): ops += 1
            else: ops += 1

        if l == r:
            if queries[l] and queries[l][0] == 2:
                ans.append(str(size[find(queries[l][1])]))
        else:
            mid = (l + r) // 2
            dfs(node * 2, l, mid)
            dfs(node * 2 + 1, mid + 1, r)

        rollback(ops)

    if q > 0: dfs(1, 1, q)

    duration = time.time() - start_time
    return (
        f"Algorithm Used: Offline Dynamic Connectivity (SegTree + DSU Rollback)\n"
        f"Execution Time: {duration:.6f} seconds\n"
        f"Answers:\n" + "\n".join(ans)
    )

In [12]:
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents.structured_output import ToolStrategy
from langchain_core.messages import HumanMessage
from pydantic import BaseModel, Field
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. تعریف قالب خروجی
class AlgorithmOutput(BaseModel):
    algorithm_name: str = Field(description="The scientific name of the algorithm (e.g., Sliding Window)")
    tool_name: str = Field(description="The exact Python function name of the tool used (e.g., solve_fence_problem)")
    execution_time: str = Field(description="Execution time in seconds")
    final_output: str = Field(description="The final calculated result")

# 2. لیست ابزارها
my_tools = [solve_fence_problem, solve_double_elimination, solve_virus_problem]

# 3. دستورات سیستم
system_prompt = """
You are an expert algorithm solver.
When a user gives you a problem description:
1. Identify which tool is appropriate.
2. CALL the tool with the correct input.
3. Extract 'Algorithm Name', 'Execution Time', and 'Final Output' from the tool's result.
4. Also, identify the 'Tool Name' (the python function name you just called).
5. Return the result ONLY using the structured format provided.
"""

# 4. ساخت ایجنت
agent = create_agent(
    model=gemini_chat,
    tools=my_tools,
    system_prompt=system_prompt,
    response_format=ToolStrategy(AlgorithmOutput)
)

# --- تابع اجرای ایجنت ---
def run_agent(user_input):

    print(f"\033[94m Agent is thinking & selecting tool...\033[0m")

    try:
        result = agent.invoke({"messages": [HumanMessage(content=user_input)]})
        structured_res = result["structured_response"]


        print("\n --- Structured Output ---")
        # اینجا نام الگوریتم و نام ابزار را کنار هم می‌گذاریم
        print(f" Algorithm: {structured_res.algorithm_name} ({structured_res.tool_name})")
        print(f" Time:      {structured_res.execution_time}")
        print(f" Result:    \n{structured_res.final_output}")
        print("--------------------------------------------------\n")
        return structured_res

    except Exception as e:
        print(f" Error: {e}")

# ==========================================
# رابط کاربری (UI)
# ==========================================
input_area = widgets.Textarea(
    value='',
    placeholder='صورت سوال را اینجا وارد کنید...',
    description='ورودی:',
    layout=widgets.Layout(width='100%', height='150px')
)

run_button = widgets.Button(description=' حل مسئله', button_style='success')
clear_button = widgets.Button(description='پاک کردن', button_style='warning')
output_area = widgets.Output(layout={'border': '1px solid #ccc', 'padding': '10px'})

def on_run_click(b):
    output_area.clear_output()
    with output_area:
        if input_area.value.strip():
            run_agent(input_area.value.strip())
        else:
            print(" لطفاً متن سوال را وارد کنید.")

def on_clear_click(b):
    input_area.value = ""
    output_area.clear_output()

run_button.on_click(on_run_click)
clear_button.on_click(on_clear_click)

print(" سیستم حل مسائل الگوریتمی (Fence, Tournament, Virus)")
display(widgets.VBox([input_area, widgets.HBox([run_button, clear_button]), output_area]))

 سیستم حل مسائل الگوریتمی (Fence, Tournament, Virus)
